In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 1 · [SETUP]  Install / upgrade all dependencies          ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"],
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])
    raise RuntimeError("pip install failed — see stderr above")
print("\n✓ All requirements installed successfully.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 2 · [VERIFY DATA]  Load, expand, engineer, split, save   ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # ensure agent/ is importable

import numpy as np
from agent.data.pipeline import DataPipeline
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir="../master_log")
logger.info("=== Phase 1 · Data Verification Start ===")

pipe = DataPipeline(splits_dir="../data/splits")

logger.info("Loading and expanding CSV …")
df = pipe.load_and_expand("../data/raw/alpha_synuclein.csv")
logger.info(f"Expanded DataFrame shape: {df.shape}")

logger.info("Engineering features (this may take ~30 s for 3-mers) …")
X, y = pipe.build_features(df)
logger.info("Feature matrix built.")

logger.info("Splitting into train / val / test …")
pipe.stratified_split(X, y, train=0.70, val=0.15, test=0.15)

logger.info("Saving splits to disk …")
pipe.save_splits()

print(f"\nTotal samples      : {len(X)}")
print(f"Feature dimensions : {X.shape[1]}")
unique, counts = np.unique(y, return_counts=True)
print(f"Class distribution : {dict(zip(unique.tolist(), counts.tolist()))}")
logger.info("=== Phase 1 · Data Verification Complete ===")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 3 · [VERIFY WALL]  Seal and verify the mathematical wall ║
# ╚══════════════════════════════════════════════════════════════════╝
from agent.core.tee_logger import TeeLogger
logger = TeeLogger()

logger.info("Sealing test set …")
pipe.seal_test_set()

logger.info("Verifying wall integrity …")
wall_intact = pipe.verify_wall()

if wall_intact:
    logger.info("Mathematical wall is INTACT. Safe to proceed.")
else:
    logger.error("WALL BREACH — test set has been tampered with!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 4 · [PHASE 2]  LLM Manager — connection test             ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from agent.core.llm_manager import LLMManager, MODELS

# ── Choose your model — change this ONE line to switch ────────────────────────
# Options: "gemini-flash" | "gemini-pro" | "groq-llama" | "groq-mixtral"
#          "mistral-small" | "cerebras" | "openrouter"
#          "local-qwen" | "local-deepseek"
llm = LLMManager(model_name="local-qwen", env_path="../.env")

# ── Show all available models ──────────────────────────────────────────────────
print("Available models:")
for alias, model_id in MODELS.items():
    marker = " <-- ACTIVE" if alias == llm.model_name else ""
    print(f"  {alias:<18} ->  {model_id}{marker}")
print()

# ── Test the connection ────────────────────────────────────────────────────────
ok = llm.test_connection()
print(f"\nConnection test: {'PASSED' if ok else 'FAILED'}")
print("If FAILED for local-qwen, try: llm.switch_model('groq-llama')")
print("Or run:  llm.auto_fallback_chain()  to find the first working model.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 5 · [PLACEHOLDER]  Phase 3: Monitor Dashboard            ║
# ╚══════════════════════════════════════════════════════════════════╝
print("Monitor dashboard will go here")